# Objectif 2 - Modelisation post-voyage

Ce notebook traite le second objectif du projet : expliquer la satisfaction client apres la realisation du sejour.

Contrairement au modele pre-voyage, cette experience peut utiliser des variables observees pendant ou apres le sejour :

- `imprevus` ;
- `reorganisation_necessaire` ;
- `respect_budget` ;
- informations extraites de `retour_client`.

Ce modele ne doit pas etre presente comme un modele de prediction avant depart. Il sert a comprendre les facteurs de satisfaction ou d'insatisfaction et a alimenter une boucle d'amelioration continue.

## 1. Configuration

On limite volontairement le nombre de threads afin de garder une execution compatible avec une machine disposant de peu de RAM.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

## 2. Chargement du dataset

Le dataset utilise reste le fichier brut fourni pour le projet. Les traitements sont reappliques dans ce notebook afin de rendre l'experience reproductible.

In [ ]:
DATA_PATH = Path("..") / "data" / "Examen_travel_planning_dataset.csv"

if not DATA_PATH.exists():
    DATA_PATH = Path("data") / "Examen_travel_planning_dataset.csv"

df_raw = pd.read_csv(DATA_PATH)
print(df_raw.shape)
display(df_raw.head())

## 3. Objectif metier et cible

Objectif post-voyage : expliquer `satisfaction_client` apres le sejour.

La cible reste le score de satisfaction client sur 5 classes :

```text
1 = tres insatisfait
5 = tres satisfait
```

Dans ce cadre, les variables post-voyage sont autorisees car le modele est retrospectif.

In [ ]:
target_column = "satisfaction_client"
post_voyage_variables = ["imprevus", "reorganisation_necessaire", "respect_budget"]
text_variable = "retour_client"

print("Cible :", target_column)
print("Variables post-voyage utilisees :", post_voyage_variables)
print("Variable texte exploitee sous forme d'indicateurs :", text_variable)

## 4. Nettoyage minimal

Les traitements restent volontairement simples et coherents avec les notebooks precedents :

- normalisation des textes en minuscules ;
- conversion des variables numeriques ;
- suppression des lignes sans cible exploitable ;
- suppression des incoherences budgetaires fortes ;
- imputation simple des valeurs manquantes.

Le but est de comparer proprement l'apport des variables post-voyage, pas de changer toute la preparation des donnees.

In [ ]:
df_model = df_raw.copy()

for column in df_model.select_dtypes(include=["object", "string"]).columns:
    df_model[column] = df_model[column].astype(str).str.strip().str.lower()

numeric_source_columns = [
    "budget_total",
    "duree_jours",
    "prix_vol",
    "satisfaction_client",
    "reorganisation_necessaire",
    "respect_budget",
]

for column in numeric_source_columns:
    df_model[column] = pd.to_numeric(df_model[column], errors="coerce")

df_model = df_model[df_model[target_column].between(1, 5)].copy()
df_model[target_column] = df_model[target_column].astype(int)

df_model = df_model[
    df_model["prix_vol"].isna()
    | df_model["budget_total"].isna()
    | (df_model["prix_vol"] <= df_model["budget_total"])
].copy()

for column in ["budget_total", "duree_jours", "prix_vol"]:
    df_model[column] = df_model[column].fillna(df_model[column].median())

categorical_columns_to_fill = [
    "client_type",
    "destination",
    "saison",
    "type_hebergement",
    "meteo_prevue",
    "activite_principale",
]

for column in categorical_columns_to_fill:
    df_model[column] = df_model[column].fillna("inconnu").replace({"nan": "inconnu"})

df_model["imprevus"] = df_model["imprevus"].fillna("aucun").replace({"nan": "aucun"})
df_model["retour_client"] = df_model["retour_client"].fillna("").replace({"nan": ""})

for column in ["reorganisation_necessaire", "respect_budget"]:
    mode_value = df_model[column].mode(dropna=True).iloc[0]
    df_model[column] = df_model[column].fillna(mode_value).astype(int)

print(df_model.shape)
display(df_model[target_column].value_counts().sort_index().rename("nombre"))

## 5. Feature engineering post-voyage

On conserve les variables pre-voyage deja creees dans les autres notebooks, puis on ajoute des variables specifiques au post-voyage.

La variable `retour_client` n'est pas injectee brute dans le modele. Elle est transformee en indicateurs simples :

- `sentiment_retour` : score lexical positif/negatif ;
- `retour_client_vide` : presence ou absence d'un commentaire.

In [ ]:
safe_duree = df_model["duree_jours"].replace(0, np.nan)
safe_budget = df_model["budget_total"].replace(0, np.nan)

df_model["budget_par_jour"] = df_model["budget_total"] / safe_duree
df_model["part_vol_budget"] = df_model["prix_vol"] / safe_budget
df_model["budget_hors_vol"] = df_model["budget_total"] - df_model["prix_vol"]
df_model["sejour_long"] = (df_model["duree_jours"] >= 14).astype(int)
df_model["meteo_risque"] = df_model["meteo_prevue"].isin(["pluie", "variable"]).astype(int)
df_model["randonnee_meteo_risque"] = (
    df_model["activite_principale"].isin(["randonn?e", "randonnee"])
    & df_model["meteo_prevue"].isin(["pluie", "variable"])
).astype(int)
df_model["saison_haute"] = df_model["saison"].isin(["?t?", "ete", "hiver"]).astype(int)
df_model["client_business"] = (df_model["client_type"] == "business").astype(int)
df_model["hebergement_luxe"] = df_model["type_hebergement"].isin(["resort", "villa"]).astype(int)

df_model["imprevu_present"] = (df_model["imprevus"] != "aucun").astype(int)
df_model["imprevu_transport"] = df_model["imprevus"].isin(["retard_vol", "annulation", "bagages"]).astype(int)
df_model["imprevu_meteo"] = df_model["imprevus"].isin(["m?t?o", "meteo"]).astype(int)
df_model["budget_non_respecte"] = (df_model["respect_budget"] == 0).astype(int)

positive_words = ["excellent", "parfait", "super", "agr?able", "agreable", "satisfait", "magnifique", "recommande"]
negative_words = ["mitig?", "mitige", "d?cevant", "decevant", "mauvais", "probl?me", "probleme", "retard", "annulation"]

def score_sentiment(text):
    text = str(text).lower()
    positive_score = sum(word in text for word in positive_words)
    negative_score = sum(word in text for word in negative_words)
    return positive_score - negative_score

df_model["sentiment_retour"] = df_model["retour_client"].apply(score_sentiment)
df_model["retour_client_vide"] = (df_model["retour_client"].str.len() == 0).astype(int)

for column in ["budget_par_jour", "part_vol_budget", "budget_hors_vol"]:
    df_model[column] = df_model[column].replace([np.inf, -np.inf], np.nan)

features_post_voyage = [
    "imprevus",
    "reorganisation_necessaire",
    "respect_budget",
    "imprevu_present",
    "imprevu_transport",
    "imprevu_meteo",
    "budget_non_respecte",
    "sentiment_retour",
    "retour_client_vide",
]

display(df_model[features_post_voyage + [target_column]].head())

## 6. Variables d'entree du modele

Dans ce notebook post-voyage, on inclut volontairement les variables observees apres ou pendant le sejour.

On exclut uniquement :

- `trip_id`, car c'est un identifiant technique ;
- `satisfaction_client`, car c'est la cible ;
- `retour_client`, car le texte brut est remplace par des indicateurs simples.

In [ ]:
excluded_columns = ["trip_id", target_column, "retour_client"]
feature_columns = [column for column in df_model.columns if column not in excluded_columns]

X = df_model[feature_columns].copy()
y = df_model[target_column].copy()

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

resume_variables = pd.DataFrame({
    "famille": ["numeriques", "categorielles", "total", "exclues"],
    "nombre": [len(numeric_features), len(categorical_features), X.shape[1], len(excluded_columns)],
    "colonnes": [numeric_features, categorical_features, feature_columns, excluded_columns],
})

display(resume_variables)
print("Variables post-voyage presentes dans X :")
print([column for column in features_post_voyage if column in X.columns])

## 7. Separation train/test et pipeline

La separation train/test reste stratifiee pour conserver la distribution des classes de satisfaction.

L'imputation, la standardisation et l'encodage sont integres dans un pipeline afin d'eviter toute fuite entre train et test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

## 8. Modelisation post-voyage

On compare un modele naif avec trois modeles simples et legers :

- `DummyClassifier` comme baseline ;
- `LogisticRegression` pour une reference interpretable ;
- `RandomForestClassifier` pour capter des relations non lineaires ;
- `ExtraTreesClassifier` comme variante d'arbres aleatoires.

La metrique principale reste `macro_f1`, car elle tient compte de toutes les classes.

In [ ]:
models = {
    "Dummy_majority": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": LogisticRegression(max_iter=500, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=42,
        class_weight="balanced",
        n_jobs=1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=120,
        max_depth=8,
        random_state=42,
        class_weight="balanced",
        n_jobs=1,
    ),
}

results = []
trained_pipelines = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    trained_pipelines[model_name] = pipeline
    results.append({
        "modele": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
    })

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(results_df.round(4))

## 9. Interpretation attendue

Cette experience doit normalement obtenir de meilleurs resultats que le modele pre-voyage, car elle utilise des informations plus proches de la satisfaction finale.

L'interpretation metier attendue est la suivante :

- les evenements reels du sejour expliquent davantage la satisfaction que les caracteristiques initiales du voyage ;
- les variables post-voyage confirment ou infirment les causes d'insatisfaction ;
- le modele est utile pour l'analyse qualite et l'amelioration continue ;
- il ne doit pas etre utilise pour faire une prediction avant depart.

In [ ]:
best_model_name = results_df.iloc[0]["modele"]
best_score = results_df.iloc[0]["macro_f1"]

print(f"Meilleur modele : {best_model_name}")
print(f"Macro F1 : {best_score:.4f}")

comparison_note = pd.DataFrame({
    "objectif": ["pre-voyage", "post-voyage"],
    "variables_post_voyage": ["exclues", "incluses"],
    "usage": ["prediction avant depart", "analyse retrospective"],
})

display(comparison_note)

## 10. Conclusion

Le modele post-voyage constitue une experience separee du modele principal pre-voyage.

Il permet de repondre a un autre besoin metier : comprendre les facteurs qui influencent la satisfaction apres le sejour. Cette approche est utile pour :

- identifier les causes d'insatisfaction ;
- suivre la qualite operationnelle ;
- ameliorer les offres futures ;
- enrichir progressivement les donnees disponibles avant depart.

La distinction entre les deux objectifs est essentielle :

- le modele pre-voyage est conforme au cas d'usage de planification, mais dispose de peu de signal ;
- le modele post-voyage contient davantage de signal, mais il est retrospectif et non deployable pour une prediction avant depart.